# Task 8B — Native physical-signal readiness

This notebook uses licensed data extracted into local runtime storage (or optionally copies it from a personal Drive folder), creates a reviewable inventory, builds the Task 8B manifest and matched TIFF view, runs source/nuisance gates, validates PRNU without binary labels, and records a fail-closed retention decision. The source images are intentionally not stored in Git: before the data preflight, stage the extracted `premier/` and `genimage_ai/` directories under `hackathon_data/raw/task8b/`, or set `CYA_TASK8B_DATA_ROOT` to a directory containing them. The notebook does not download source data, modify RINE, read the competition final test, or enable chromatic aberration automatically.

In [ ]:
from pathlib import Path
import subprocess

def checkout_present(path):
    return (path / 'configs/colab.json').is_file() and (path / 'Makefile').is_file()

cwd = Path.cwd()
running_from_checkout = checkout_present(cwd) or checkout_present(cwd.parent)
cloud_checkout = Path('/content/cya-techjam26')
if not running_from_checkout and Path('/content').is_dir():
    if (cloud_checkout / '.git').is_dir():
        subprocess.run(['git', '-C', str(cloud_checkout), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(
            ['git', 'clone', 'https://github.com/maxi-cmyk/cya-techjam26.git', str(cloud_checkout)],
            check=True,
        )

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

def find_project_root():
    configured = os.environ.get('CYA_PROJECT_ROOT')
    starts = [Path(configured)] if configured else []
    starts.extend([Path.cwd(), Path.cwd().parent, Path('/content/cya-techjam26')])
    checked = set()
    for start in starts:
        for candidate in (start, *start.parents):
            resolved = candidate.resolve()
            if resolved in checked:
                continue
            checked.add(resolved)
            if (resolved / 'configs/colab.json').is_file() and (resolved / 'Makefile').is_file():
                return resolved
    raise RuntimeError(
        'Could not locate the cya-techjam26 checkout. Open this notebook from the repository, '
        'or set CYA_PROJECT_ROOT to its absolute path.'
    )

PROJECT_ROOT = find_project_root()
SOURCE_MODE = 'local'  # 'local' or 'private_drive'
PRIVATE_DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/cya-techjam26-data')
TASK8B_DATA_ROOT_OVERRIDE = os.environ.get('CYA_TASK8B_DATA_ROOT', '').strip()
if TASK8B_DATA_ROOT_OVERRIDE:
    LOCAL_TASK8B = Path(TASK8B_DATA_ROOT_OVERRIDE).expanduser().resolve()
    LOCAL_DATA_ROOT = LOCAL_TASK8B.parents[1]
else:
    LOCAL_DATA_ROOT = (
        Path('/content/hackathon_data')
        if PROJECT_ROOT == Path('/content/cya-techjam26')
        else PROJECT_ROOT / 'hackathon_data'
    )
    LOCAL_TASK8B = LOCAL_DATA_ROOT / 'raw/task8b'
LOCAL_ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
PRIVATE_DRIVE_TASK8B = PRIVATE_DRIVE_DATA_ROOT / 'raw/task8b'

assert SOURCE_MODE in {'local', 'private_drive'}, 'Invalid SOURCE_MODE'
print('Project root:', PROJECT_ROOT)
print('Task 8B data root:', LOCAL_TASK8B)

In [ ]:
if SOURCE_MODE == 'private_drive':
    if not PRIVATE_DRIVE_TASK8B.is_dir():
        raise FileNotFoundError(
            f'Missing personal Drive data: {PRIVATE_DRIVE_TASK8B}. Mount Drive or update '
            'PRIVATE_DRIVE_DATA_ROOT before continuing.'
        )
    LOCAL_TASK8B.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(PRIVATE_DRIVE_TASK8B, LOCAL_TASK8B, dirs_exist_ok=True)

required_directories = [LOCAL_TASK8B / 'premier', LOCAL_TASK8B / 'genimage_ai']
missing_directories = [path for path in required_directories if not path.is_dir()]
if missing_directories:
    missing = '\n'.join(f'  - {path}' for path in missing_directories)
    raise FileNotFoundError(
        'The code checkout is ready, but Task 8B image data is not included in Git.\n'
        f'Missing directories:\n{missing}\n\n'
        'Stage the extracted data at those paths, or set CYA_TASK8B_DATA_ROOT to a '
        'directory that directly contains premier/ and genimage_ai/. Only the extracted '
        'PREMIER sample (about 1.9 GB) and Tiny-GenImage sample (about 44 MB) are needed; '
        'the original PREMIER tar.gz files are not required.'
    )
print('Using Task 8B source data at:', LOCAL_TASK8B)

In [ ]:
environment = os.environ.copy()
environment['DATA_ROOT'] = str(LOCAL_DATA_ROOT)
environment['ARTIFACT_ROOT'] = str(LOCAL_ARTIFACT_ROOT)
inventory = LOCAL_TASK8B / 'sources.csv'
if not inventory.is_file():
    subprocess.run(['make', 'task8b-inventory'], cwd=PROJECT_ROOT, env=environment, check=True)
    raise RuntimeError(
        'A draft sources.csv was created. Review it and inventory_preparation.json, '
        'correct any metadata, then rerun this cell.'
    )
subprocess.run(['make', 'task8b-prepare'], cwd=PROJECT_ROOT, env=environment, check=True)

In [ ]:
readiness_path = LOCAL_ARTIFACT_ROOT / 'task8b/audits/readiness_report.json'
readiness = json.loads(readiness_path.read_text(encoding='utf-8'))
print(json.dumps({key: readiness[key] for key in ('source_ready', 'training_ready', 'prnu_reference', 'chromatic_aberration')}, indent=2))
assert readiness['source_ready'], 'Source readiness failed; inspect readiness_report.json'
if readiness['prnu_reference']['ready']:
    subprocess.run(['make', 'task8b-prnu-references'], cwd=PROJECT_ROOT, env=environment, check=True)
else:
    print('PRNU references remain blocked by device/image coverage.')
subprocess.run(['make', 'task8b-matched'], cwd=PROJECT_ROOT, env=environment, check=True)
subprocess.run(['make', 'task8b-prnu-validate'], cwd=PROJECT_ROOT, env=environment, check=True)
subprocess.run(['make', 'task8b-decision'], cwd=PROJECT_ROOT, env=environment, check=True)
decision_path = LOCAL_ARTIFACT_ROOT / 'task8b/reports/retention_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
print(json.dumps(decision, indent=2))
if not decision['fusion_training_eligible']:
    print('Task 8B is complete with no fusion run; no physical estimator passed its independent gate.')

In [ ]:
local_results = LOCAL_ARTIFACT_ROOT / 'task8b'
drive_results = DRIVE_ARTIFACT_ROOT / 'task8b'
if Path('/content/drive/MyDrive').is_dir():
    drive_results.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
    print('Synced Task 8B artifacts to:', drive_results)
else:
    print('Drive is not mounted; artifacts remain at:', local_results)